# CSMorgan v9 — SimulaMet Published Decoding Parameters

**Hypothesis:** v1 underperforms SimulaMet's published 85.78% eval accuracy because we use greedy decoding (`temperature=0`, `top_k=1`, `max_tokens=20`) while SimulaMet's published example uses:

```python
RequestConfig(max_tokens=512, temperature=0.3, top_k=20, top_p=0.7,
              repetition_penalty=1.05)
```

**Experiment design:** take v1 unchanged, patch ONLY the `RequestConfig` line in `submission_task1.py`. Reuse v1's `normalization.py` + `answer_bank.json` + everything else. Push as v9. Validate.

**Outcomes:**
- `rouge1 >= 0.65` → decoding was the bottleneck. Submit v9.
- `rouge1 ≈ 0.55` (within noise of v1's 0.5423) → decoding parameters don't help on this validation set. We learn nothing major and move on to v10 (Transf model).
- `rouge1 < 0.50` → decoding parameters hurt (unlikely but possible if sampling introduces too much variance). Fall back to v1.

**Time:** ~15 min total (5 min setup + 10 min validate).
**Risk:** very low. We're not training, not changing the model, not modifying normalization. Just swapping 5 numbers in one config call.

### 1. Install + HF login

In [ ]:
%%capture
!pip install -q -U huggingface_hub medvqa
from huggingface_hub import notebook_login, whoami, HfApi, hf_hub_download, create_repo
notebook_login()

In [ ]:
info = whoami()
assert info['name'] == 'sageofai', f'Logged in as {info["name"]}'
print('Logged in as:', info['name'])

Logged in as: sageofai


### 2. Fetch v1's files, patch decoding, push to v9

In [ ]:
import os, re
from huggingface_hub import hf_hub_download, HfApi, create_repo

SRC = 'sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1'        # v1 (best baseline)
DST = 'sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9'     # decoding experiment

api = HfApi()
create_repo(DST, repo_type='model', exist_ok=True, private=False)
print(f'Target repo ready: https://huggingface.co/{DST}')

Target repo ready: https://huggingface.co/sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9


In [ ]:
# Pull v1's submission_task1.py
v1_script_path = hf_hub_download(repo_id=SRC, filename='submission_task1.py', repo_type='model')
with open(v1_script_path) as f:
    code = f.read()
print(f'Loaded v1 submission_task1.py ({len(code)} chars)')

# Show the current RequestConfig block so we can verify the patch worked
import re
m = re.search(r'RequestConfig\(([^)]+)\)', code)
if m:
    print('\nCurrent RequestConfig:')
    print('  RequestConfig(' + m.group(1).strip() + ')')
else:
    print('WARNING: did not find RequestConfig(...) in v1 script. Will need manual patch.')

submission_task1.py:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

Loaded v1 submission_task1.py (14551 chars)

Current RequestConfig:
  RequestConfig(max_tokens=32,         # short answers only - 1-3 words typical
    temperature=0.0,       # greedy
    top_k=1,
    top_p=1.0,
    repetition_penalty=1.0,)


In [ ]:
# Patch the RequestConfig to SimulaMet's published parameters
SIMULAMET_REQ_CFG = '''RequestConfig(
    max_tokens=512,
    temperature=0.3,
    top_k=20,
    top_p=0.7,
    repetition_penalty=1.05,
)'''

# Replace using regex - matches RequestConfig(...) including newlines inside
patched, n = re.subn(
    r'RequestConfig\([^)]*\)',
    SIMULAMET_REQ_CFG.replace('\n', '\n'),  # keep newlines
    code,
    count=1,
    flags=re.DOTALL,
)

assert n == 1, f'Expected exactly 1 RequestConfig replacement, did {n}'

# Also patch repo references inside the script so it loads v9 if it self-references
patched = patched.replace(SRC, DST)

# Update SUBMISSION_INFO Notes_to_organizers if present
patched = re.sub(
    r'("Notes_to_organizers"\s*:\s*)\([^)]*\)',
    r'\1("v9: v1 baseline with SimulaMet-recommended decoding params "'
    r'"(temp=0.3, top_k=20, top_p=0.7, rep_penalty=1.05, max_tokens=512). "'
    r'"All other components (adapter, normalization, answer-bank) identical to v1.")',
    patched,
    count=1,
    flags=re.DOTALL,
)

# Optional: bump BLEU to max_order=1 if it isn't already
if 'max_order=1' not in patched:
    old = 'bleu_result = bleu.compute(predictions=preds, references=references)'
    new = (
        'bleu_result = bleu.compute(predictions=preds, references=references, max_order=1)\n'
        '    bleu4_result = bleu.compute(predictions=preds, references=references)'
    )
    patched = patched.replace(old, new)

# Save locally
with open('/content/submission_task1.py', 'w') as f:
    f.write(patched)

# Show the new RequestConfig for visual confirmation
m = re.search(r'RequestConfig\(([^)]+)\)', patched, re.DOTALL)
print('Patched RequestConfig:')
print('  RequestConfig(' + m.group(1).strip() + ')')
print(f'\nFinal script: {len(patched)} chars')

Patched RequestConfig:
  RequestConfig(max_tokens=512,
    temperature=0.3,
    top_k=20,
    top_p=0.7,
    repetition_penalty=1.05,)

Final script: 14498 chars


In [ ]:
# Copy v1's companion files (normalization.py, answer_bank.json, requirements.txt) into v9
# These don't change — we want exactly v1's inference behavior except decoding.
for fname in ['normalization.py', 'answer_bank.json', 'requirements.txt']:
    try:
        local = hf_hub_download(repo_id=SRC, filename=fname, repo_type='model')
        api.upload_file(
            path_or_fileobj=local, path_in_repo=fname,
            repo_id=DST, repo_type='model',
            commit_message=f'v9: copy {fname} from v1 (unchanged)',
        )
        print(f'  ✓ {fname}')
    except Exception as e:
        print(f'  ⚠ skipping {fname}: {e}')

# Upload patched submission_task1.py
api.upload_file(
    path_or_fileobj='/content/submission_task1.py',
    path_in_repo='submission_task1.py',
    repo_id=DST, repo_type='model',
    commit_message='v9: SimulaMet-recommended decoding (temp=0.3, top_k=20, top_p=0.7, rep_pen=1.05, max_tokens=512)',
)
print('  ✓ submission_task1.py')
print(f'\n✅ v9 ready at https://huggingface.co/{DST}')

normalization.py:   0%|          | 0.00/39.2k [00:00<?, ?B/s]

  ✓ normalization.py


answer_bank.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...a84de02c/answer_bank.json:  18%|#8        | 10.6MB / 58.2MB            

  ✓ answer_bank.json


requirements.txt:   0%|          | 0.00/672 [00:00<?, ?B/s]

  ✓ requirements.txt
  ✓ submission_task1.py

✅ v9 ready at https://huggingface.co/sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9


In [ ]:
from huggingface_hub import hf_hub_download, HfApi

SRC = 'sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1'        # v1
DST = 'sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9'     # this experiment
api = HfApi()

# Fetch v1's requirements.txt
req_path = hf_hub_download(repo_id=SRC, filename='requirements.txt', repo_type='model')
with open(req_path) as f:
    reqs = f.read()
print('Current requirements.txt:')
print(reqs)

# Add hf_transfer if missing
if 'hf_transfer' not in reqs:
    reqs = reqs.rstrip() + '\nhf_transfer>=0.1.6\n'
    print('\n✓ Added hf_transfer')
else:
    print('\n  hf_transfer already in requirements')

# Also add it as a defensive runtime install at the top of submission_task1.py
# in case the container's pip-install phase fails to pick it up
sub_path = hf_hub_download(repo_id=DST, filename='submission_task1.py', repo_type='model')
with open(sub_path) as f:
    code = f.read()

defensive_prefix = '''# Defensive: ensure hf_transfer is available (medvqa container sets HF_HUB_ENABLE_HF_TRANSFER=1)
import subprocess, sys
try:
    import hf_transfer  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer"], check=False)
    # If still missing, just disable the env var so datasets falls back to normal download
    try:
        import hf_transfer  # noqa: F401
    except ImportError:
        import os
        os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)

'''

if 'hf_transfer' not in code.split('\n', 30)[0:30][0] if False else 'defensive: ensure hf_transfer' not in code.lower():
    # Insert near the top, after the docstring/shebang but before normal imports
    # Find the first import line
    lines = code.split('\n')
    insert_idx = 0
    for i, ln in enumerate(lines[:20]):
        if ln.startswith('import ') or ln.startswith('from '):
            insert_idx = i
            break
    new_code = '\n'.join(lines[:insert_idx]) + '\n' + defensive_prefix + '\n'.join(lines[insert_idx:])
else:
    new_code = code

# Verify it still parses
import ast
try:
    ast.parse(new_code)
    print('✓ Patched submission_task1.py is valid Python')
except SyntaxError as e:
    print(f'❌ SyntaxError at line {e.lineno}: {e.msg}')
    raise

# Write both files locally
with open('/content/requirements.txt', 'w') as f:
    f.write(reqs)
with open('/content/submission_task1.py', 'w') as f:
    f.write(new_code)

# Push both
api.upload_file(
    path_or_fileobj='/content/requirements.txt',
    path_in_repo='requirements.txt',
    repo_id=DST, repo_type='model',
    commit_message='v9 fix: add hf_transfer to requirements',
)
api.upload_file(
    path_or_fileobj='/content/submission_task1.py',
    path_in_repo='submission_task1.py',
    repo_id=DST, repo_type='model',
    commit_message='v9 fix: defensive hf_transfer install at runtime',
)
print(f'\n✅ Pushed fixes to https://huggingface.co/{DST}')
print(f'\nRe-run: !medvqa validate --competition=gi-2026 --task=1 --repo_id={DST}')

Current requirements.txt:
# ImageCLEFmed-MEDVQA-GI-2026 Task 1 — CSMorgan-MEDVQA
# Pinned to the Kvasir-VQA-x1 reference setup (ms-swift 3.8.0 + qwen_vl_utils 0.0.11)
# so that PtEngine + Qwen/Qwen2.5-VL-7B-Instruct + the QLoRA adapter
# (sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1) load consistently inside the
# medvqa validate_and_submit container.

# --- model runtime ---
torch>=2.1.0
transformers>=4.45.0
accelerate>=0.34.0
peft>=0.13.0
bitsandbytes>=0.43.0
ms-swift==3.8.0
qwen_vl_utils==0.0.11

# --- data + evaluation ---
datasets>=2.20.0
evaluate>=0.4.3
sacrebleu>=2.4.0
rouge_score>=0.1.2
nltk>=3.9.0

# --- utilities ---
huggingface_hub>=0.24.0
Pillow>=10.0.0
tqdm>=4.66.0
numpy>=1.26.0


✓ Added hf_transfer
✓ Patched submission_task1.py is valid Python

✅ Pushed fixes to https://huggingface.co/sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9

Re-run: !medvqa validate --competition=gi-2026 --task=1 --repo_id=sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9


### 3. Validate (~10 min on medvqa container)

In [ ]:
!medvqa validate --competition=gi-2026 --task=1 --repo_id=sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9

🌟 ImageCLEFmed-MEDVQA-GI-2026 🌟 https://github.com/simula/ImageCLEFmed-MEDVQA-GI-2026
🔍 Subtask 1: Clinically Relevant Visual Question Answering
👀 Analyzing submission repository: sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9 👀
Logged in to HuggingFace as: sageofai
Loaded as API: https://simulamet-medvqa-gi-2026.hf.space ✔
💓 Communicating with the Submission Server: Ping!
Pong! Submission server is alive! 😊
Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
requirements.txt: 100% 691/691 [00:00<00:00, 5.51MB/s]
Fetching 2 files:  50% 1/2 [00:00<00:00,  1.37it/s]
submission_task1.py: 15.1kB [00:00, 17.0MB/s]
Fetching 2 files: 100% 2/2 [00:00<00:00,  2.62it/s]
📦 Making sure of the minimum requirements to run the script 📦
📦 Installing requirements from the submission repo: sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9/requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.5 MB/s eta 0:00:00
2026-05-24 21:50:43.906181: I tensorflow/core/util/port.cc:153] oneDNN custom operations ar

### 4. Length-distribution diagnostic

After validate, check that predictions still match the reference distribution. With `max_tokens=512` removed as a cap, the model COULD produce longer outputs — verify it still produces short canonical answers like v1.

In [ ]:
import json, numpy as np
from pathlib import Path
from collections import Counter
from datasets import load_dataset

pred_path = max(Path('/root/.cache/huggingface/hub').rglob('predictions_1.json'),
                key=lambda p: p.stat().st_mtime, default=None)
if pred_path is None:
    print('predictions_1.json not found — run validate first.')
else:
    data = json.load(open(pred_path))
    raw = data.get('predictions', data) if isinstance(data, dict) else data
    if raw and isinstance(raw[0], dict):
        preds = [str(p.get('answer', '')) for p in raw]
    else:
        preds = [str(p) for p in raw]

    ds = load_dataset('SimulaMet/Kvasir-VQA-test', split='validation')
    def _flat(r): return '; '.join(str(x) for x in r) if isinstance(r, list) else str(r)
    refs = [_flat(ex['answer']) for ex in ds]

    def wc(x): return len(str(x).strip().split())
    pred_lens = [wc(p) for p in preds]
    ref_lens  = [wc(r) for r in refs]

    print('=== PRED vs REF distribution ===')
    print(f'  ref avg / pred avg     : {np.mean(ref_lens):.2f} / {np.mean(pred_lens):.2f}')
    print(f'  ref 1-word % / pred 1-word % : '
          f'{100*sum(l==1 for l in ref_lens)/len(ref_lens):.1f} / '
          f'{100*sum(l==1 for l in pred_lens)/len(pred_lens):.1f}')
    print(f'  ref >5 words % / pred >5 words % : '
          f'{100*sum(l>5 for l in ref_lens)/len(ref_lens):.1f} / '
          f'{100*sum(l>5 for l in pred_lens)/len(pred_lens):.1f}')
    print(f'\n  ref length histo : {Counter(ref_lens).most_common(5)}')
    print(f'  pred length histo: {Counter(pred_lens).most_common(5)}')

    print('\n=== VERDICT ===')
    pred_avg, ref_avg = np.mean(pred_lens), np.mean(ref_lens)
    if pred_avg > ref_avg * 1.8:
        print('⚠  Predictions much longer than refs - v9 decoding triggered prose drift.')
    elif abs(pred_avg - ref_avg) < 0.5:
        print('✓  Length distribution matches v1 / refs. Decoding change did not cause drift.')
    else:
        print('~  Some length difference, watch the scores.')

    # Show top-20 predictions for sanity
    print('\nTop-20 predicted answers:')
    for a, c in Counter(preds).most_common(20):
        print(f'  {c:5d}  {a!r}')

predictions_1.json not found — run validate first.


### 5. Decision: submit or move to v10?

Look at Phase 3's `Public scores`:
- `rouge1 >= 0.55` → submit v9 (Phase 6 below)
- `rouge1 ≈ 0.52-0.54` → noise. Move to v10 (Transf model swap) before submitting.
- `rouge1 < 0.50` → v9 regressed (sampling variance maybe). Don't submit, go back to v1.

In [ ]:
CONFIRM = True
if CONFIRM:
    !medvqa validate_and_submit --competition=gi-2026 --task=1 --repo_id=sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9
else:
    print('CONFIRM=False. Only set True if Phase 3 rouge1 clearly beats v1 (0.5423).')

🌟 ImageCLEFmed-MEDVQA-GI-2026 🌟 https://github.com/simula/ImageCLEFmed-MEDVQA-GI-2026
🔍 Subtask 1: Clinically Relevant Visual Question Answering
👀 Analyzing submission repository: sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9 👀
Logged in to HuggingFace as: sageofai
Loaded as API: https://simulamet-medvqa-gi-2026.hf.space ✔
💓 Communicating with the Submission Server: Ping!
Pong! Submission server is alive! 😊
Fetching 2 files: 100% 2/2 [00:00<00:00, 26214.40it/s]
📦 Making sure of the minimum requirements to run the script 📦
📦 Installing requirements from the submission repo: sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v9/requirements.txt
2026-05-24 22:41:52.548984: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-24 22:41:52.620072: I tensorflow/core/platform/cpu_feature_gu

### 6. Fallback: resubmit v1 if v9 didn't beat it